# Eigenfaces and PCA for Face Recognition

This project explores dimensionality reduction and face recognition using
Principal Component Analysis (PCA), eigenfaces, and a Random Forest classifier
on the **Labeled Faces in the Wild (LFW)** dataset.

The workflow reduces high-dimensional face images into a compact PCA feature
space and evaluates how well those features support identity classification.

## Project Overview

The project includes:

- Loading and visualising the LFW face dataset
- Splitting data into training and testing sets
- Mean-centering face images
- Computing PCA components using Singular Value Decomposition (SVD)
- Visualising eigenfaces
- Analysing cumulative explained variance
- Projecting faces into a lower-dimensional feature space
- Training a Random Forest classifier on PCA features
- Evaluating predictions on held-out test data


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

print("NumPy version:", np.__version__)


NumPy version: 2.3.0


## 1. Dataset

The **Labeled Faces in the Wild (LFW)** dataset is used for face-recognition
experiments.

To keep the classification task focused on identities with sufficient examples,
the dataset is loaded with a minimum number of images per person and resized for
efficient computation.


In [ ]:
lfw_people = fetch_lfw_people(min_faces_per_person=70, resize=0.4)

n_samples, h, w = lfw_people.images.shape

X = lfw_people.data
y = lfw_people.target
target_names = lfw_people.target_names

n_features = X.shape[1]
n_classes = target_names.shape[0]

print("Total dataset size:")
print("n_samples:", n_samples)
print("n_features:", n_features)
print("n_classes:", n_classes)
print("image shape:", (h, w))
print("classes:")
for i, name in enumerate(target_names):
    print(f"  {i}: {name}")


NameError: name 'fetch_lfw_people' is not defined

## 2. Example Face Images

A selection of face images is visualised to inspect the dataset and the
variation in pose, illumination, expression, and appearance.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
axes = axes.ravel()

for i, ax in enumerate(axes):
    ax.imshow(lfw_people.images[i], cmap="gray")
    ax.set_title(target_names[y[i]], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 3. Train/Test Split

The dataset is divided into training and testing subsets.

The training set is used to learn the PCA representation and train the
classifier, while the test set is kept separate for evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


## 4. Mean Face and Data Centering

PCA is applied to centred data.

The mean face is computed using the training set and subtracted from both
training and testing samples. This ensures that the principal components model
variation around the average training face.


In [ ]:
mean_face = np.mean(X_train, axis=0)

X_train_centered = X_train - mean_face
X_test_centered = X_test - mean_face

plt.figure(figsize=(3, 4))
plt.imshow(mean_face.reshape(h, w), cmap="gray")
plt.title("Mean Face")
plt.axis("off")
plt.show()


## 5. PCA and Eigenfaces using SVD

Principal Component Analysis is computed using Singular Value Decomposition
(SVD).

The leading principal directions capture the dominant patterns of variation in
the face dataset. When these PCA basis vectors are reshaped back into image
form, they are commonly referred to as **eigenfaces**.

The first 150 principal components are retained for the lower-dimensional face
representation.


In [ ]:
n_components = 150

U, S, V = np.linalg.svd(X_train_centered, full_matrices=False)

components = V[:n_components]
eigenfaces = components.reshape((n_components, h, w))

print("U shape:", U.shape)
print("S shape:", S.shape)
print("V shape:", V.shape)
print("components shape:", components.shape)
print("eigenfaces shape:", eigenfaces.shape)


## 6. Projection into PCA Face Space

Each centred face image is projected from the original pixel space into the
lower-dimensional PCA feature space.

This representation reduces dimensionality while preserving a large proportion
of the variation present in the original training data.


In [ ]:
X_transformed = np.dot(X_train_centered, components.T)
X_test_transformed = np.dot(X_test_centered, components.T)

print("Original training shape:", X_train.shape)
print("PCA training shape:", X_transformed.shape)
print("Original testing shape:", X_test.shape)
print("PCA testing shape:", X_test_transformed.shape)


## 7. Eigenface Visualisation

The leading PCA components are visualised as eigenfaces.

These images represent major directions of variation learned from the training
faces rather than individual identities.


In [ ]:
def plot_gallery(images, titles, h, w, n_row=3, n_col=4):
    plt.figure(figsize=(1.8 * n_col, 2.4 * n_row))
    plt.subplots_adjust(bottom=0, left=0.01, right=0.99, top=0.90, hspace=0.35)

    for i in range(n_row * n_col):
        plt.subplot(n_row, n_col, i + 1)
        plt.imshow(images[i].reshape((h, w)), cmap=plt.cm.gray)
        plt.title(titles[i], size=12)
        plt.xticks(())
        plt.yticks(())

eigenface_titles = [f"eigenface {i}" for i in range(eigenfaces.shape[0])]
plot_gallery(eigenfaces, eigenface_titles, h, w)
plt.show()


## 8. Explained Variance

Cumulative explained variance is used to measure how much information is
retained as additional PCA components are included.

The curve illustrates the trade-off between dimensionality reduction and
variance preservation.


In [ ]:
explained_variance = (S ** 2) / (n_samples - 1)
total_var = explained_variance.sum()
explained_variance_ratio = explained_variance / total_var
ratio_cumsum = np.cumsum(explained_variance_ratio)

eigenvalue_count = np.arange(n_components)

plt.figure(figsize=(8, 5))
plt.plot(eigenvalue_count, ratio_cumsum[:n_components])
plt.title("Compactness")
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance ratio")
plt.grid(True)
plt.show()

print(
    f"Cumulative variance captured by {n_components} components:",
    ratio_cumsum[n_components - 1]
)


## 9. Face Classification with Random Forest

A Random Forest classifier is trained using the PCA-transformed face features.

This provides a classical machine-learning baseline for identity classification
before moving to convolutional neural networks in the next experiment.


In [ ]:
estimator = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    max_features=150,
    random_state=42
)

estimator.fit(X_transformed, y_train)
predictions = estimator.predict(X_test_transformed)

correct = predictions == y_test
total_test = len(X_test_transformed)
accuracy = np.sum(correct) / total_test

print("Total Testing:", total_test)
print("Total Correct:", np.sum(correct))
print("Accuracy:", accuracy)

print()
print("Classification Report:")
print(classification_report(
    y_test,
    predictions,
    target_names=target_names
))


## 10. Classification Results

A verified run produced a test accuracy of approximately **60.56%** using the
PCA + Random Forest pipeline.

The classification report shows that performance varies between identities,
which is expected because the classes contain different numbers of images and
different levels of visual variation.


## 11. Example Predictions

Example predictions are displayed with the true identity and the classifier's
predicted identity.

These examples provide a qualitative view of both successful predictions and
common failure cases.


In [ ]:
n_show = 12
fig, axes = plt.subplots(3, 4, figsize=(10, 9))
axes = axes.ravel()

for i, ax in enumerate(axes[:n_show]):
    image = X_test[i].reshape(h, w)
    true_name = target_names[y_test[i]]
    pred_name = target_names[predictions[i]]

    ax.imshow(image, cmap="gray")
    ax.set_title(f"True: {true_name}\nPred: {pred_name}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 12. Key Findings

- PCA provides a compact representation of high-dimensional face images.
- Eigenfaces visualise the dominant directions of variation learned from the
  training data.
- The PCA feature space enables classification using a conventional machine
  learning model.
- The verified Random Forest baseline reached approximately **60.56%** test
  accuracy.
- The result provides a useful baseline for comparison with the CNN-based face
  classifier.

## Technologies

- Python
- NumPy
- Matplotlib
- scikit-learn
- PCA / SVD
- Random Forest
- Computer Vision

## Skills Demonstrated

- Dimensionality reduction
- Singular Value Decomposition
- Eigenface analysis
- Feature-space visualisation
- Classical machine-learning classification
- Model evaluation and interpretation

## Project Context

This work was developed as part of **COMP3710** at
**The University of Queensland**.
